In [2]:
import torch
import cv2
import pandas as pd
import numpy as np
import os
from transformers import BlipForConditionalGeneration, AutoProcessor
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
MODEL_DIR = os.path.join(BASE_PATH, "blip_video_model_2")
DATA_CSV = os.path.join(BASE_PATH, "task4_domains.csv") # Reusing the tiny dataset!
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

def get_video_frames(video_path, num_frames=4):
    """Safely extracts frames and handles the path issue."""
    vid_path = video_path.replace("project 2", "video_captioning")
    cap = cv2.VideoCapture(vid_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(0, total_frames - 1), num_frames, dtype=int)
    
    frames = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)))
        else:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
    cap.release()
    return frames

# --- 1. FGSM Adversarial Attack (Instructions 1 & 2) ---
def fgsm_attack(model, processor, frame, target_caption, epsilon=0.05):
    """Adds invisible mathematical noise to trick the model."""
    inputs = processor(images=frame, text=target_caption, return_tensors="pt", padding="max_length", max_length=20)
    pixel_values = inputs.pixel_values.to(DEVICE)
    input_ids = inputs.input_ids.to(DEVICE)
    
    # Enable gradient tracking for the image so we can "hack" it
    pixel_values.requires_grad = True
    
    # THE FIX: Explicitly pass input_ids along with pixel_values and labels!
    outputs = model(
        pixel_values=pixel_values, 
        input_ids=input_ids, 
        labels=input_ids
    )
    
    model.zero_grad()
    outputs.loss.backward()
    
    # Create the adversarial image by adding the sign of the gradient
    data_grad = pixel_values.grad.data
    perturbed_image = pixel_values + epsilon * data_grad.sign()
    
    # Generate new caption from the hacked image
    with torch.no_grad():
        out_ids = model.generate(pixel_values=perturbed_image, max_length=20)
    
    return processor.decode(out_ids[0], skip_special_tokens=True)

# --- 2. Optical Flow Extraction (Instruction 5) ---
def get_optical_flow(frame1, frame2):
    """Converts standard RGB video into motion vectors (Optical Flow)."""
    prvs = cv2.cvtColor(frame1, cv2.COLOR_RGB2GRAY)
    next_f = cv2.cvtColor(frame2, cv2.COLOR_RGB2GRAY)
    
    flow = cv2.calcOpticalFlowFarneback(prvs, next_f, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

# --- MAIN EXPERIMENT LOOP ---
def run_task5():
    print("Loading Model for Robustness Testing...")
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
    model.eval()
    
    df = pd.read_csv(DATA_CSV).head(3) # Just test on 3 videos to prove the concept!
    smoothie = SmoothingFunction().method4
    
    for idx, row in df.iterrows():
        print(f"\n=========================================")
        print(f"🛡️ TESTING VIDEO: {os.path.basename(row['video_path'])}")
        print(f"Ground Truth: '{row['caption']}'")
        print(f"=========================================")
        
        frames = get_video_frames(row['video_path'], num_frames=4)
        base_frame = frames[1] # Use an early frame for the baseline
        
        # --- TEST A: Standard Baseline ---
        inputs = processor(images=base_frame, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            clean_out = model.generate(**inputs, max_length=20)
        clean_caption = processor.decode(clean_out[0], skip_special_tokens=True)
        print(f"✅ Baseline (Clean RGB): {clean_caption}")
        
        # --- TEST B: FGSM Attack (Instruction 1 & 2) ---
        adv_caption = fgsm_attack(model, processor, base_frame, row['caption'], epsilon=0.1)
        print(f"😈 FGSM Adversarial (Epsilon 0.1): {adv_caption}")
        
        # --- TEST C: Modality Dropout / Masking (Instruction 3 & 4) ---
        print("\n  ▶ Frame Masking (Modality Dropout) Test:")
        dropouts = [0, 1, 2] # 0%, 25%, 50% of 4 frames
        percentages = ["0%", "25%", "50%"]
        
        for drop_count, pct in zip(dropouts, percentages):
            test_frames = frames.copy()
            for i in range(drop_count):
                test_frames[i] = np.zeros((224, 224, 3), dtype=np.uint8) # Mask with black frame
                
            inputs = processor(images=test_frames, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                mask_out = model.generate(**inputs, max_length=20)
            
            # Average BLEU across the processed frames
            bleus = []
            for out in mask_out:
                pred = processor.decode(out, skip_special_tokens=True).split()
                ref = [row['caption'].split()]
                bleus.append(sentence_bleu(ref, pred, smoothing_function=smoothie))
                
            print(f"    ↳ {pct} Masked -> Avg BLEU Score: {np.mean(bleus):.4f}")

        # --- TEST D: Extreme Scenario - Optical Flow (Instruction 5) ---
        flow_frame = get_optical_flow(frames[0], frames[1])
        inputs = processor(images=flow_frame, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            flow_out = model.generate(**inputs, max_length=20)
        flow_caption = processor.decode(flow_out[0], skip_special_tokens=True)
        print(f"🌊 Extreme Scenario (Optical Flow Only): {flow_caption}")

if __name__ == "__main__":
    run_task5()

Loading Model for Robustness Testing...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 10689.72it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



🛡️ TESTING VIDEO: video50.mp4
Ground Truth: 'a girl and boy flirt then eat food'


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


✅ Baseline (Clean RGB): a man is talking about a computer
😈 FGSM Adversarial (Epsilon 0.1): a man is lifting weights

  ▶ Frame Masking (Modality Dropout) Test:
    ↳ 0% Masked -> Avg BLEU Score: 0.0304
    ↳ 25% Masked -> Avg BLEU Score: 0.0294
    ↳ 50% Masked -> Avg BLEU Score: 0.0294
🌊 Extreme Scenario (Optical Flow Only): a man is talking about a child s voice

🛡️ TESTING VIDEO: video50.mp4
Ground Truth: 'a guy and a girl sitting on a sofa look at each while their mouths are full of food'
✅ Baseline (Clean RGB): a man is talking about a computer
😈 FGSM Adversarial (Epsilon 0.1): a man is lifting weights

  ▶ Frame Masking (Modality Dropout) Test:
    ↳ 0% Masked -> Avg BLEU Score: 0.0093
    ↳ 25% Masked -> Avg BLEU Score: 0.0096
    ↳ 50% Masked -> Avg BLEU Score: 0.0096
🌊 Extreme Scenario (Optical Flow Only): a man is talking about a child s voice

🛡️ TESTING VIDEO: video50.mp4
Ground Truth: 'a man twirls around then has food stuffed into his mouth as a girl glares at him'
✅ Bas